# Taller 2 — Modelo 1: Expected Goals (xG)
## Regresión Logística
**Machine Learning I (ML1-2026I) — Universidad Externado de Colombia**

import pandas as pd
import numpy as np
import requests
import pickle
import json
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

Configuración de directorios
ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

API_BASE = "https://premier.72-60-245-2.sslip.io"

── 1. Cargar eventos (tiros) ────────────────────────────────────────────
print("⬇️  Cargando eventos/tiros desde el API ...")
ruta_shots_raw = DATA_DIR / "shots_raw.csv"

if ruta_shots_raw.exists():
    print("   Usando caché local de tiros.")
    events = pd.read_csv(ruta_shots_raw)
else:
    print("   Descargando de API...")
    resp = requests.get(f"{API_BASE}/events?is_shot=true&limit=10000", timeout=60)
    resp.raise_for_status()
    shots_raw = resp.json().get("events", [])
    events = pd.DataFrame(shots_raw)
    events.to_csv(ruta_shots_raw, index=False)
    print(f"   {len(events)} tiros descargados.")

Filtrar solo tiros si es necesario
if 'is_shot' in events.columns:
    shots = events[events['is_shot'] == True].copy()
else:
    shots = events.copy()

print(f"   Total tiros: {len(shots)}")

── 2. Feature Engineering ──────────────────────────────────────────────
print("🔧 Generando features de tiros ...")

Limpiar qualifiers (contiene información en JSON/string de las acciones del tiro)
shots['q_str'] = shots['qualifiers'].fillna('[]').astype(str)

Normalizar coordenadas: si x < 50, estaban atacando el arco izquierdo (x=0).
Reflejamos x e y para que SIEMPRE asuman que atacan el arco derecho (x=100).
shots['x_norm'] = np.where(shots['x'] < 50, 100 - shots['x'], shots['x'])
shots['y_norm'] = np.where(shots['x'] < 50, 100 - shots['y'], shots['y'])

Coordenadas: x va de 0 a 100 (largo), y va de 0 a 100 (ancho). 
El arco suele estar en x=100, y=50.
shots['distance'] = np.sqrt((100 - shots['x_norm'])**2 + (50 - shots['y_norm'])**2)
shots['angle']    = np.degrees(np.arctan2(50 - shots['y_norm'], 100 - shots['x_norm']))
shots['y_offset'] = abs(shots['y_norm'] - 50)

-- LIMPIEZA DE OUTLIERS Y AUTOGOLES --
Evitamos que autogoles y tiros mayores a 50 metros distorsionen los coeficientes
shots = shots[~shots['q_str'].str.contains('OwnGoal', na=False)].copy()
shots = shots[shots['distance'] <= 50].copy()

Referencia a la serie de texto limpia
q = shots['q_str']

Extraer features booleanas del campo qualifiers
shots['is_big_chance'] = q.str.contains('BigChance',    na=False).astype(int)
shots['is_penalty']    = q.str.contains('"Penalty"',    na=False).astype(int)
shots['is_header']     = q.str.contains('"Head"',       na=False).astype(int)
shots['is_right_foot'] = q.str.contains('RightFoot',    na=False).astype(int)
shots['is_left_foot']  = q.str.contains('LeftFoot',     na=False).astype(int)
shots['is_counter']    = q.str.contains('FastBreak',    na=False).astype(int)
shots['from_corner']   = q.str.contains('FromCorner',   na=False).astype(int)
shots['is_volley']     = q.str.contains('Volley',       na=False).astype(int)
shots['first_touch']   = q.str.contains('FirstTouch',   na=False).astype(int)
shots['is_assisted']   = q.str.contains('IntentionalAssist|KeyPass', na=False).astype(int)
shots['zone_box_centre']  = q.str.contains('BoxCentre',      na=False).astype(int)
shots['zone_out_of_box']  = q.str.contains('OutOfBox',        na=False).astype(int)
shots['zone_small_box']   = q.str.contains('SmallBox',        na=False).astype(int)

Variable objetivo
if 'is_goal' not in shots.columns and 'isGoal' in shots.columns:
    shots['is_goal'] = shots['isGoal']
    
y = shots['is_goal'].astype(int)

Definir features a usar en el modelo
FEATURES_XG = [
    'distance', 'angle',
    'is_big_chance', 'is_penalty', 'is_header', 'is_right_foot', 'is_left_foot',
    'is_counter', 'is_assisted', 'is_volley',
    'zone_box_centre', 'zone_out_of_box', 'zone_small_box'
]

X = shots[FEATURES_XG].fillna(0)

print(f"Dataset xG listo: {X.shape[0]} tiros × {X.shape[1]} features")
print(f"Distribución is_goal:\n{y.value_counts(normalize=True)}")

── 3. Modelado (Regresión Logística) ──────────────────────────────────
print("\n⚙️ Entrenando modelo de Regresión Logística...")

scaler_xg = StandardScaler()
X_scaled = scaler_xg.fit_transform(X)

Logistic Regression con class_weight='balanced' debido a que los goles son pocos (~11%)
model_xg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

Evaluación con Cross-Validation k=5
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    model_xg, X_scaled, y, cv=kf,
    scoring=['accuracy', 'roc_auc', 'f1', 'recall'],
    return_train_score=False
)

auc_cv = cv_results['test_roc_auc'].mean()
acc_cv = cv_results['test_accuracy'].mean()
recall_cv = cv_results['test_recall'].mean()
f1_cv = cv_results['test_f1'].mean()

print("\nResultados Cross-Validation (k=5):")
print(f"  AUC-ROC : {auc_cv:.4f} ± {cv_results['test_roc_auc'].std():.4f}")
print(f"  Accuracy: {acc_cv:.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"  Recall  : {recall_cv:.4f} ± {cv_results['test_recall'].std():.4f}")
print(f"  F1-Score: {f1_cv:.4f} ± {cv_results['test_f1'].std():.4f}")

Entrenar en todo el dataset para guardar
model_xg.fit(X_scaled, y)
y_pred = model_xg.predict(X_scaled)
y_prob = model_xg.predict_proba(X_scaled)[:, 1]

print("\nReporte en Train Completo:")
print(classification_report(y, y_pred, target_names=['No Goal', 'Goal']))

print("\nTop Features (Coeficientes de Regresión Logística):")
coef_df = pd.DataFrame({
    'Feature': FEATURES_XG,
    'Coeficiente': model_xg.coef_[0]
}).sort_values(by='Coeficiente', key=abs, ascending=False)
print(coef_df.head(10).to_string(index=False))

── 4. Guardar Artefactos ──────────────────────────────────────────────
print("\n💾 Guardando modelos en carpeta 'models/'...")

with open(MODELS_DIR / "model_xg.pkl", "wb") as f:
    pickle.dump(model_xg, f)
with open(MODELS_DIR / "scaler_xg.pkl", "wb") as f:
    pickle.dump(scaler_xg, f)
with open(MODELS_DIR / "features_xg.pkl", "wb") as f:
    pickle.dump(FEATURES_XG, f)

print("✅ Modelo xG guardado exitosamente.")